# Preparation TRENDY data

In [ ]:
# Libraries
import os
import gzip
import shutil
import xarray as xr

In [ ]:
# Import functions to calculate grid cell area and grid cell boundaries
from xgrid_utils import grid_cell_areas

In [ ]:
# Directories
dir_trendyv12 = '../TRENDY/trendyv12-gcb2023/'
dir07 = '../paper_ecosystem_carbon/output/07_eval_prep/'

---

### Get Trendy data

In [ ]:
def get_trendyv12_data(dir_trendyv12, m):
    """
    Function to check for 'S3' directory in the given directory `m`,
    and load files containing 'cveg' and 'csoil' using xarray if they exist.
    Handles both `.nc` and `.nc.gz` files and resolves time decoding issues.

    Parameters:
        dir_trendyv12 (str): Path to the main directory containing subdirectories.
        m (str): Name of the subdirectory to process.

    Returns:
        xarray.Dataset: Merged dataset with renamed variables and dimensions.
                        None if 'S3' directory or required files are missing.
    """
    # Check if the directory contains an 'S3' subdirectory
    s3_path = os.path.join(dir_trendyv12, m, 'S3')
    
    mstr = m.lower()
    
    if not os.path.isdir(s3_path):
        print(f'No S3 directory found in: {m}')
        return None

    # Search for files containing 'cveg' and 'csoil' in the S3 directory
    cveg_file = next((f for f in os.listdir(s3_path) if '_cVeg.' in f and 'S3' in f), None)
    csoil_file = next((f for f in os.listdir(s3_path) if '_cSoil.' in f and 'S3' in f), None)

    # If either file is missing, return None
    if not cveg_file or not csoil_file:
        print(f'Missing cveg or csoil files in S3 directory of: {m}')
        return None

    # Load the data using xarray
    try:
        cveg_path = os.path.join(s3_path, cveg_file)
        csoil_path = os.path.join(s3_path, csoil_file)

        # Handle .nc.gz files by decompressing them temporarily
        def decompress_gz(file_path):
            if file_path.endswith('.gz'):
                decompressed_path = file_path[:-3]  # Remove .gz extension
                with gzip.open(file_path, 'rb') as f_in:
                    with open(decompressed_path, 'wb') as f_out:
                        shutil.copyfileobj(f_in, f_out)
                return decompressed_path
            return file_path

        # Decompress if necessary
        cveg_decompressed = decompress_gz(cveg_path)
        csoil_decompressed = decompress_gz(csoil_path)

        # Load datasets with xarray
        def safe_open(file_path):
            try:
                # First try with automatic time decoding
                return xr.open_dataset(file_path, chunks='auto')
            except (ValueError, TypeError):
                # Fallback to no time decoding
                return xr.open_dataset(file_path, chunks='auto', decode_times=False)

        cveg_data = safe_open(cveg_decompressed)
        csoil_data = safe_open(csoil_decompressed)
        # Rename variables
        cveg_data = cveg_data.rename(cVeg=f'{mstr}_cveg')
        csoil_data = csoil_data.rename(cSoil=f'{mstr}_csoil')
        
        # Merge data
        cdata = xr.merge([cveg_data, csoil_data])

        # Add Model name to attributes
        for i in cdata.data_vars:
            cdata[i].attrs['original_model_name'] = m
        
        print(f'Successfully loaded data: {m}')
        
        # Clean up temporary decompressed files if they were created
        if cveg_decompressed != cveg_path and os.path.exists(cveg_decompressed):
            os.remove(cveg_decompressed)
        if csoil_decompressed != csoil_path and os.path.exists(csoil_decompressed):
            os.remove(csoil_decompressed)
        
        return cdata
    except Exception as e:
        print(f'Error loading data: {m}')
        return None


ds_cablepop = get_trendyv12_data(dir_trendyv12, 'CABLEPOP')
# ds_cardamom = get_trendyv12_data(dir_trendyv12, 'CARDAMOM') # time: Monthly time step given in days since 01/01/2003
ds_classic = get_trendyv12_data(dir_trendyv12, 'CLASSIC')
ds_clm = get_trendyv12_data(dir_trendyv12, 'CLM5.0')
ds_dlem = get_trendyv12_data(dir_trendyv12, 'DLEM')
#ds_e3sm = get_trendyv12_data(dir_trendyv12, 'E3SM') # Montly data
ds_ed = get_trendyv12_data(dir_trendyv12, 'ED')
# ds_elm = get_trendyv12_data(dir_trendyv12, 'ELM') # no s3 run
ds_ibis = get_trendyv12_data(dir_trendyv12, 'IBIS')
ds_isam = get_trendyv12_data(dir_trendyv12, 'ISAM')
ds_isbactrip = get_trendyv12_data(dir_trendyv12, 'ISBACTRIP') # Montly data
ds_jsbach = get_trendyv12_data(dir_trendyv12, 'JSBACH')
ds_jules = get_trendyv12_data(dir_trendyv12, 'JULES')
ds_lpjguess = get_trendyv12_data(dir_trendyv12, 'LPJ-GUESS')
ds_LPJmL = get_trendyv12_data(dir_trendyv12, 'LPJmL')
#ds_lpjwsl = get_trendyv12_data(dir_trendyv12, 'LPJwsl') # Error loading, check!
ds_lpxqs = get_trendyv12_data(dir_trendyv12, 'lpxqs')
ds_ocn = get_trendyv12_data(dir_trendyv12, 'OCN')
ds_orchidee = get_trendyv12_data(dir_trendyv12, 'ORCHIDEE')
ds_sdgvm = get_trendyv12_data(dir_trendyv12, 'SDGVM')
ds_visit = get_trendyv12_data(dir_trendyv12, 'VISIT') # Montly data
ds_yibs = get_trendyv12_data(dir_trendyv12, 'YIBS')

---

### Fix inconsistencies

In [ ]:
# Helper function to prepare grid cell area array
def prep_da_area(ds):
    """Prepare DataArray for grid cell areas."""
    return xr.DataArray(
        data=grid_cell_areas(ds.lon.data, ds.lat.data),
        dims=('lat', 'lon'),
        coords={'lat': ds.lat.data, 'lon': ds.lon.data}
    ).chunk(lat=ds.chunks['lat'], lon=ds.chunks['lon']) * 0.0001

# Helper function to fix attributes of summarized data arrays
def fix_attrs(ds, ds_ysum):
    """ Copies attributes from ds variables to ds_ysum variables and changes unit attribute"""
    for i in ds.data_vars:
        ds_ysum[i].attrs = ds[i].attrs
        ds_ysum[i].attrs['units'] = 'PgC'
    return ds_ysum

# Functions to fix inconsistencies for each model data
def prep_ds_cablepop_ysum():
    """Prepare cablepop dataset for yearly summation."""
    ds = ds_cablepop.copy()
    ds = ds.rename(dict(latitude='lat', longitude='lon'))
    ds['time'] = (ds['time'] + 1700).astype(int)
    da_area = prep_da_area(ds)
    ds_ysum = ((ds * 10 * da_area).sum(['lat', 'lon']) * 1E-9).compute()
    ds_ysum = fix_attrs(ds, ds_ysum)
    return ds_ysum

def prep_ds_jsbach_ysum():
    """Prepare jsbach dataset for yearly summation."""
    ds = ds_jsbach.copy()
    ds = ds.rename(dict(latitude='lat', longitude='lon'))
    ds = ds.groupby('time.year').mean('time').rename(dict(year='time')).chunk('auto')
    da_area = prep_da_area(ds)
    ds_ysum = ((ds * 10 * da_area).sum(['lat', 'lon']) * 1E-9).compute()
    ds_ysum = fix_attrs(ds, ds_ysum)
    return ds_ysum

def prep_ds_classic_ysum():
    """Prepare classic dataset for yearly summation."""
    ds = ds_classic.copy()
    ds = ds.rename(dict(latitude='lat', longitude='lon'))
    ds['time'] = [dt.year for dt in ds.time.values]
    da_area = prep_da_area(ds)
    ds_ysum = ((ds * 10 * da_area).sum(['lat', 'lon']) * 1E-9).compute()
    ds_ysum = fix_attrs(ds, ds_ysum)
    return ds_ysum

def prep_ds_clm_ysum():
    """Prepare clm dataset for yearly summation."""
    ds = ds_clm.copy()
    ds = ds.rename({'clm5.0_cveg': 'clm_cveg','clm5.0_csoil': 'clm_csoil'})
    ds['time'] = [dt.year for dt in ds.time.values]
    da_area = prep_da_area(ds)
    ds_ysum = ((ds * 10 * da_area).sum(['lat', 'lon']) * 1E-9).compute()
    ds_ysum = fix_attrs(ds, ds_ysum)
    return ds_ysum

def prep_ds_dlem_ysum():
    """Prepare dlem dataset for yearly summation."""
    ds = ds_dlem.copy()
    ds['time'] = (ds['time'] + 1700).astype(int)
    da_area = prep_da_area(ds)
    ds_ysum = ((ds * 10 * da_area).sum(['lat', 'lon']) * 1E-9).compute()
    ds_ysum = fix_attrs(ds, ds_ysum)
    return ds_ysum

def prep_ds_ibis_ysum():
    """Prepare ibis dataset for yearly summation."""
    ds = ds_ibis.copy()
    ds = ds.rename(dict(latitude='lat', longitude='lon'))
    ds['time'] = (ds['time'] + 1700).astype(int)
    da_area = prep_da_area(ds)
    ds_ysum = ((ds * 10 * da_area).sum(['lat', 'lon']) * 1E-9).compute()
    ds_ysum = fix_attrs(ds, ds_ysum)
    return ds_ysum

def prep_ds_isam_ysum():
    """Prepare isam dataset for yearly summation."""
    ds = ds_isam.copy()
    da_area = prep_da_area(ds)
    ds_ysum = ((ds * 10 * da_area).sum(['lat', 'lon']) * 1E-9).compute()
    ds_ysum = fix_attrs(ds, ds_ysum)
    return ds_ysum

def prep_ds_jules_ysum():
    """Prepare jules dataset for yearly summation."""
    ds = ds_jules.copy()
    ds['time'] = [dt.year for dt in ds.time.values]
    da_area = prep_da_area(ds)
    ds_ysum = ((ds * 10 * da_area).sum(['lat', 'lon']) * 1E-9).compute()
    ds_ysum = fix_attrs(ds, ds_ysum)
    return ds_ysum

def prep_ds_lpjguess_ysum():
    """Prepare lpjguess dataset for yearly summation."""
    ds = ds_lpjguess.copy()
    ds = ds.rename(dict(latitude='lat', longitude='lon'))
    ds['time'] = [dt.year for dt in ds.time.values]
    da_area = prep_da_area(ds)
    ds_ysum = ((ds * 10 * da_area).sum(['lat', 'lon']) * 1E-9).compute()
    ds_ysum = fix_attrs(ds, ds_ysum)
    return ds_ysum

def prep_ds_LPJmL_ysum():
    """Prepare LPJmL dataset for yearly summation."""
    ds = ds_LPJmL.copy()
    ds = ds.rename(dict(latitude='lat', longitude='lon'))
    ds['time'] = (ds['time'] + 1700).astype(int)
    da_area = prep_da_area(ds)
    ds_ysum = ((ds * 10 * da_area).sum(['lat', 'lon']) * 1E-9).compute()
    ds_ysum = fix_attrs(ds, ds_ysum)
    return ds_ysum

def prep_ds_lpxqs_ysum():
    """Prepare lpxqs dataset for yearly summation."""
    ds = ds_lpxqs.copy()
    ds = ds.rename(dict(latitude='lat', longitude='lon'))
    ds['time'] = [dt.year for dt in ds.time.values]
    da_area = prep_da_area(ds)
    ds_ysum = ((ds * 10 * da_area).sum(['lat', 'lon']) * 1E-9).compute()
    ds_ysum = fix_attrs(ds, ds_ysum)
    return ds_ysum

def prep_ds_ocn_ysum():
    """Prepare ocn dataset for yearly summation."""
    ds = ds_ocn.copy()
    ds = ds.rename(dict(latitude='lat', longitude='lon'))
    ds['time'] = [dt.year for dt in ds.time.values]
    da_area = prep_da_area(ds)
    ds_ysum = ((ds * 10 * da_area).sum(['lat', 'lon']) * 1E-9).compute()
    ds_ysum = fix_attrs(ds, ds_ysum)
    return ds_ysum

def prep_ds_orchidee_ysum():
    """Prepare orchidee dataset for yearly summation."""
    ds = ds_orchidee.copy()
    ds = ds.rename(dict(latitude='lat', longitude='lon'))
    ds['time'] = [dt.year for dt in ds.time.values]
    da_area = prep_da_area(ds)
    ds_ysum = ((ds * 10 * da_area).sum(['lat', 'lon']) * 1E-9).compute()
    ds_ysum = fix_attrs(ds, ds_ysum)
    return ds_ysum

def prep_ds_sdgvm_ysum():
    """Prepare sdgvm dataset for yearly summation."""
    ds = ds_sdgvm.copy()
    ds = ds.rename(dict(latitude='lat', longitude='lon'))
    ds['time'] = [dt.year for dt in ds.time.values]
    da_area = prep_da_area(ds)
    ds_ysum = ((ds * 10 * da_area).sum(['lat', 'lon']) * 1E-9).compute()
    ds_ysum = fix_attrs(ds, ds_ysum)
    return ds_ysum

def prep_ds_yibs_ysum():
    """Prepare yibs dataset for yearly summation."""
    ds = ds_yibs.copy()
    ds = ds.rename(dict(latitude='lat', longitude='lon'))
    ds['time'] = (ds['time'] + 1700).astype(int)
    da_area = prep_da_area(ds)
    ds_ysum = ((ds * 10 * da_area).sum(['lat', 'lon']) * 1E-9).compute()
    ds_ysum = fix_attrs(ds, ds_ysum)
    return ds_ysum


# Fix inconsistencies and merge datasets
ds_trendy_ysum = xr.merge([
    prep_ds_cablepop_ysum(),
    prep_ds_jsbach_ysum(),
    prep_ds_classic_ysum(),
    prep_ds_clm_ysum(),
    prep_ds_dlem_ysum(),
    prep_ds_ibis_ysum(),
    prep_ds_isam_ysum(),
    prep_ds_jules_ysum(),
    prep_ds_lpjguess_ysum(),
    prep_ds_LPJmL_ysum(),
    prep_ds_lpxqs_ysum(),
    prep_ds_ocn_ysum(),
    prep_ds_orchidee_ysum(),
    prep_ds_sdgvm_ysum(),
    prep_ds_yibs_ysum()
])

# Fix lpxbern name issues
ds_trendy_ysum = ds_trendy_ysum.rename({'lpxqs_cveg': "lpxbern_cveg",
                                       'lpxqs_csoil': "lpxbern_csoil"})
ds_trendy_ysum.lpxbern_cveg.attrs['original_model_name'] = 'LPX-Bern'
ds_trendy_ysum.lpxbern_csoil.attrs['original_model_name'] = 'LPX-Bern'

In [ ]:
# Add trendy ensemble mean for cveg and csoil to dataset
def add_trendy_mean():
    """Adds ensemble mean of cveg and csoil to ds_trendy_ysum"""    
    for ctype in ['cveg', 'csoil']:
        l_vars = [i for i in ds_trendy_ysum.data_vars if i.endswith(ctype)]
        da_mean = ds_trendy_ysum[l_vars].to_array().mean('variable')
        
        if ctype == 'cveg':
            long_name = 'Carbon in Vegetation'
        if ctype == 'csoil':
            long_name = 'Carbon in Soil'
        
        da_mean.attrs = {'long_name': long_name,
                    'units': 'PgC',
                    'original_model_name': 'TRENDY mean'}
        
        ds_trendy_ysum[f'trendymean_{ctype}'] = da_mean
        

add_trendy_mean()

In [ ]:
# Export
ds_trendy_ysum.to_netcdf(f'{dir07}ds_trendy_ysum.nc', mode='w')